In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

# Load dataset
raw_data = pd.read_csv('/kaggle/input/bhaav-dataset/bhaav.csv')
train_val, test = train_test_split(raw_data, test_size=0.2, random_state=42)
train, val = train_test_split(train_val, test_size=0.1, random_state=42)
dataset = DatasetDict({
    'train': Dataset.from_pandas(train),
    'validation': Dataset.from_pandas(val),
    'test': Dataset.from_pandas(test),
})

# Define number of labels (assuming 4 emotions, adjust as needed)
num_labels = 4

# Load tokenizer and model
model_name = '/kaggle/input/xlm-roberta-base/xlm-roberta-base' # Start anew from xlm-roberta-base
# model_name = '/kaggle/working/bhavnai-model' # Resume from last trained model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Preprocessing function for the tokenizer
def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

# Tokenize the dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Define training arguments
training_args = TrainingArguments(
    save_strategy='epoch',
    output_dir='./bhaavnai-output',
    logging_strategy='epoch',
    logging_dir='./logs',
    eval_strategy='epoch',
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    report_to='none',
    fp16=True,
    warmup_steps=500,
    lr_scheduler_type='cosine',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    resume_from_checkpoint=True,
)

# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {
        'accuracy': acc,
        'f1': f1,
    }

# Weighted loss Trainer subclass
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_cpu = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')

        # Ensure class weights are on the same device as the inputs/model
        effective_class_weights = None
        if self.class_weights_cpu is not None:
             effective_class_weights = self.class_weights_cpu.to(inputs['input_ids'].device)

        # Forward pass
        outputs = model(**inputs)
        logits = outputs.logits

        # Compute custom loss
        loss_fct = torch.nn.CrossEntropyLoss(weight=effective_class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# Compute class weights from the train set
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(dataset['train']['label']),
    y=dataset['train']['label']
)
class_weights = torch.tensor(class_weights).float()

# Define Trainer
trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Train the model
trainer.train()

# Save model & tokenizer
trainer.save_model('/kaggle/working/bhaavnai-model')
tokenizer.save_pretrained('/kaggle/working/bhaavnai-model')

In [ ]:
from sklearn.metrics import classification_report
preds_output = trainer.predict(tokenized_dataset['test'])
print(classification_report(preds_output.label_ids, np.argmax(preds_output.predictions, axis=1)))
print(pd.DataFrame(trainer.evaluate()))

In [ ]:
import re
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

# Load your fine-tuned model and tokenizer
model_path = './bhavnai-model'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path, attn_implementation='eager')

# Define your label mapping (index to label)
labels = ['Angry 😠', 'Sad 😢', 'Neutral 😐', 'Happy 😊']

# Input text
input_text = 'graahaka sahaayataa kitnii bekaara hai yaahaa!'

processed_text = transliterate(input_text, sanscript.ITRANS, sanscript.DEVANAGARI)

# Tokenize input
inputs = tokenizer(processed_text, return_tensors='pt', truncation=True, padding=True)

# Predict
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)
    probs = F.softmax(outputs.logits, dim=-1)
    attentions = outputs.attentions

    # Calculate the average attention across all layers and heads
    attentions_avg = torch.mean(attentions[-1], dim=(1, 2))  # Using the last layer for simplicity

    # Map the attention scores to token positions
    attention_scores = attentions_avg.squeeze().detach().cpu().numpy()

    # Get token ids and convert them back to text
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze().cpu().numpy())

    # Extract keywords from the processed text based on attention values of tokens
    keywords = {kw: 0 for kw in re.findall('[।.,?!:;"\'\\(\\)\\-]|[^।.,?!:;"\'\\(\\)\\-\\s]+', processed_text) if len(kw) != 0}
    for t, a in zip(tokens, attention_scores):
        for kw in keywords.keys():
            if t.lstrip('▁') in kw:
                keywords[kw] += a
                break
    if len(keywords) != 0:
        keywords = [kw for kw, _ in sorted(keywords.items(), key=lambda item: item[1], reverse=True)]

predicted_output = {
    'text': processed_text,
    'sentiment': labels[probs.argmax().item()],
    'confidence': probs.max().item(),
    'keywords': keywords[:5] if keywords else [],
}

print(f'Predicted output: {predicted_output}')